# 🧹 Signal Cleaning: From Noisy to Crystal Clear

Welcome to the world of signal preprocessing! Real biosignals are never perfect - they're always contaminated with noise.

## What You'll Learn Today

In 60 minutes, you'll:
- Understand different types of noise in biosignals
- Apply various filters to clean signals
- Remove artifacts (unwanted patterns)
- Compare before and after results
- Learn when to use which filtering technique

## Skills You'll Master

- **Low-pass filtering** - removing high-frequency noise
- **High-pass filtering** - removing baseline drift
- **Band-pass filtering** - keeping only desired frequencies
- **Notch filtering** - removing specific interference (like 50/60 Hz power line)
- **Artifact removal** - eliminating motion and eye blink artifacts

**Ready to become a signal cleaning expert? Let's clean up!** 🧽

## Step 1: Load Signal Processing Tools

We'll need some powerful tools for this job!

In [ ]:
# Import signal processing toolkit
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import fft, fftfreq
from scipy.signal import butter, filtfilt, iirnotch, find_peaks
import warnings
warnings.filterwarnings('ignore')

# Make professional plots
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✅ Signal cleaning tools loaded!")
print("🧹 Ready to make noisy signals crystal clear!")

---
# 📊 Understanding Noise in Biosignals

## Types of Noise and Interference

Real-world biosignals face many challenges:

### 1. High-Frequency Noise 📡
- **Source**: Electronic interference, muscle activity (EMG)
- **Appearance**: Fast, random wiggles on top of the signal
- **Solution**: Low-pass filter

### 2. Baseline Drift (Low-Frequency) 🌊
- **Source**: Movement, breathing, sweat changing electrode contact
- **Appearance**: Slow up-and-down wandering
- **Solution**: High-pass filter

### 3. Power Line Interference ⚡
- **Source**: Electrical wiring (50 Hz in Europe, 60 Hz in USA)
- **Appearance**: Regular oscillation at exactly 50 or 60 Hz
- **Solution**: Notch filter

### 4. Artifacts 💥
- **Source**: Eye blinks, movement, electrode pops
- **Appearance**: Large, sudden spikes or drops
- **Solution**: Artifact detection and removal

Let's create noisy signals and learn how to clean them!

In [ ]:
def generate_clean_ecg(duration=10, heart_rate=75, sampling_rate=500):
    """
    Generate a clean ECG signal (our ground truth)
    """
    time = np.linspace(0, duration, int(duration * sampling_rate))
    signal_clean = np.zeros(len(time))
    
    beat_interval = 60 / heart_rate
    current_time = 0.5
    
    while current_time < duration - 1:
        beat_index = int(current_time * sampling_rate)
        beat_length = int(0.6 * sampling_rate)
        
        if beat_index + beat_length < len(signal_clean):
            t = np.linspace(0, 1, beat_length)
            
            # PQRST complex
            p_wave = 0.25 * np.exp(-((t - 0.2) ** 2) / 0.005)
            qrs = 1.8 * np.exp(-((t - 0.35) ** 2) / 0.0008)
            t_wave = 0.35 * np.exp(-((t - 0.6) ** 2) / 0.01)
            
            beat = p_wave + qrs + t_wave
            signal_clean[beat_index:beat_index + beat_length] += beat
        
        current_time += beat_interval + np.random.uniform(-0.02, 0.02)
    
    return time, signal_clean

def add_noise(clean_signal, sampling_rate=500, 
              high_freq_noise=0.1, baseline_drift=0.2, 
              powerline=True, artifacts=False):
    """
    Add various types of noise to a clean signal
    """
    noisy_signal = clean_signal.copy()
    time = np.arange(len(clean_signal)) / sampling_rate
    
    # 1. High-frequency noise (muscle activity, electronics)
    if high_freq_noise > 0:
        hf_noise = np.random.normal(0, high_freq_noise, len(clean_signal))
        noisy_signal += hf_noise
    
    # 2. Baseline drift (breathing, movement)
    if baseline_drift > 0:
        # Mix of slow drifts at different frequencies
        drift = baseline_drift * np.sin(2 * np.pi * 0.15 * time)
        drift += baseline_drift * 0.5 * np.sin(2 * np.pi * 0.3 * time)
        noisy_signal += drift
    
    # 3. Power line interference (50 or 60 Hz)
    if powerline:
        powerline_freq = 60  # Hz (use 50 for Europe)
        powerline_noise = 0.15 * np.sin(2 * np.pi * powerline_freq * time)
        noisy_signal += powerline_noise
    
    # 4. Artifacts (sudden spikes)
    if artifacts:
        num_artifacts = np.random.randint(2, 5)
        for _ in range(num_artifacts):
            artifact_pos = np.random.randint(0, len(noisy_signal) - 100)
            artifact_amplitude = np.random.uniform(1.5, 3.0)
            artifact_sign = np.random.choice([-1, 1])
            # Create a spike artifact
            artifact_len = 50
            artifact = artifact_sign * artifact_amplitude * np.exp(-np.linspace(0, 5, artifact_len))
            noisy_signal[artifact_pos:artifact_pos + artifact_len] += artifact
    
    return noisy_signal

print("✅ Noise generation functions ready!")

## Let's See What Noise Does to Our Signal

We'll create a clean ECG and then add all types of noise to see the damage!

In [ ]:
# Generate clean signal
duration = 10
sampling_rate = 500
time, clean_ecg = generate_clean_ecg(duration=duration, heart_rate=75, sampling_rate=sampling_rate)

# Add ALL the noise!
noisy_ecg = add_noise(clean_ecg, sampling_rate=sampling_rate,
                      high_freq_noise=0.1, 
                      baseline_drift=0.2,
                      powerline=True,
                      artifacts=True)

# Compare clean vs noisy
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10))

# Clean signal
zoom_time = 5  # Show 5 seconds
zoom_mask = time <= zoom_time

ax1.plot(time[zoom_mask], clean_ecg[zoom_mask], linewidth=2, color='green', label='Clean ECG')
ax1.set_ylabel('Voltage (mV)', fontsize=12)
ax1.set_title('✅ Clean ECG Signal (Original)', fontsize=14, fontweight='bold', color='green')
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=11)
ax1.set_xlim(0, zoom_time)

# Noisy signal
ax2.plot(time[zoom_mask], noisy_ecg[zoom_mask], linewidth=1.5, color='red', label='Noisy ECG', alpha=0.8)
ax2.set_xlabel('Time (seconds)', fontsize=12)
ax2.set_ylabel('Voltage (mV)', fontsize=12)
ax2.set_title('❌ Noisy ECG Signal (With All Types of Noise)', fontsize=14, fontweight='bold', color='red')
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=11)
ax2.set_xlim(0, zoom_time)

plt.tight_layout()
plt.show()

print("\n🔍 Can you see the problems in the noisy signal?")
print("   • Fast wiggles (high-frequency noise)")
print("   • Slow up-and-down drift (baseline wander)")
print("   • Regular ripples (60 Hz power line)")
print("   • Sudden large spikes (artifacts)")
print("\n🎯 Our mission: Remove ALL this noise and recover the clean signal!")

---
# 🔧 Tool #1: Low-Pass Filter (Remove High-Frequency Noise)

## What is a Low-Pass Filter?

A **low-pass filter** lets low frequencies pass through while blocking high frequencies. 

Think of it like a gate:
- **Lets through**: Slow changes (the actual ECG waves)
- **Blocks**: Fast wiggles (noise from muscles and electronics)

## Key Parameter: Cutoff Frequency

The cutoff frequency decides what's "low" vs "high":
- For ECG: typical cutoff is 40-50 Hz
- Frequencies below cutoff → pass through
- Frequencies above cutoff → removed

## The Butterworth Filter

We'll use a **Butterworth filter** - it's smooth and well-behaved. The "order" controls how sharp the cutoff is (higher order = sharper cutoff).

Let's implement it!

In [ ]:
def apply_lowpass_filter(signal_data, cutoff_freq, sampling_rate=500, order=4):
    """
    Apply a low-pass Butterworth filter
    
    cutoff_freq: frequency in Hz above which to remove
    order: filter order (higher = sharper cutoff)
    """
    # Design the filter
    nyquist = sampling_rate / 2  # Nyquist frequency
    normal_cutoff = cutoff_freq / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    
    # Apply the filter (filtfilt applies forward and backward for zero phase shift)
    filtered_signal = filtfilt(b, a, signal_data)
    
    return filtered_signal

# Create a signal with ONLY high-frequency noise
_, clean_test = generate_clean_ecg(duration=5, sampling_rate=sampling_rate)
noisy_test = add_noise(clean_test, sampling_rate=sampling_rate,
                       high_freq_noise=0.15,  # High-frequency noise only
                       baseline_drift=0,
                       powerline=False,
                       artifacts=False)

# Apply low-pass filter
filtered_test = apply_lowpass_filter(noisy_test, cutoff_freq=40, sampling_rate=sampling_rate)

# Compare results
time_test = np.arange(len(clean_test)) / sampling_rate

fig, axes = plt.subplots(3, 1, figsize=(16, 11))

# Original clean
axes[0].plot(time_test[:1500], clean_test[:1500], 'g-', linewidth=2, label='Clean (Original)')
axes[0].set_ylabel('Voltage (mV)', fontsize=11)
axes[0].set_title('✅ Original Clean Signal', fontsize=13, fontweight='bold', color='green')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Noisy
axes[1].plot(time_test[:1500], noisy_test[:1500], 'r-', linewidth=1.5, alpha=0.7, label='Noisy')
axes[1].set_ylabel('Voltage (mV)', fontsize=11)
axes[1].set_title('❌ With High-Frequency Noise', fontsize=13, fontweight='bold', color='red')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

# Filtered
axes[2].plot(time_test[:1500], filtered_test[:1500], 'b-', linewidth=2, label='After Low-Pass Filter')
axes[2].plot(time_test[:1500], clean_test[:1500], 'g--', linewidth=1.5, alpha=0.5, label='Original (reference)')
axes[2].set_xlabel('Time (seconds)', fontsize=11)
axes[2].set_ylabel('Voltage (mV)', fontsize=11)
axes[2].set_title('✅ After Low-Pass Filter (40 Hz cutoff)', fontsize=13, fontweight='bold', color='blue')
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()

print("\n🎉 Success! The low-pass filter removed the fast wiggles!")
print("   The filtered signal (blue) closely matches the clean original (green).")

## 💡 Try This!

**Experiment with different cutoff frequencies**:
1. Go back and change `cutoff_freq=40` to `cutoff_freq=20`
   - What happens? (Hint: too much filtering removes real signal!)
2. Try `cutoff_freq=60`
   - What happens? (Hint: not enough filtering leaves noise!)
3. Find the "sweet spot" for best results!

---
# 🔧 Tool #2: High-Pass Filter (Remove Baseline Drift)

## What is a High-Pass Filter?

A **high-pass filter** does the opposite - it lets high frequencies through while blocking low frequencies.

- **Lets through**: Fast changes (the heartbeat features)
- **Blocks**: Slow drifts (breathing, movement, sweat)

## When to Use It

Use high-pass filtering when:
- Signal baseline wanders up and down
- You see slow "waves" underneath the real signal
- Recording is affected by movement or breathing

## Key Parameter: Cutoff Frequency

For ECG, typical high-pass cutoff is 0.5-1 Hz:
- Removes very slow drifts
- Keeps heartbeat frequencies (1-40 Hz)

Let's try it!

In [ ]:
def apply_highpass_filter(signal_data, cutoff_freq, sampling_rate=500, order=4):
    """
    Apply a high-pass Butterworth filter
    
    cutoff_freq: frequency in Hz below which to remove
    """
    nyquist = sampling_rate / 2
    normal_cutoff = cutoff_freq / nyquist
    b, a = butter(order, normal_cutoff, btype='high', analog=False)
    filtered_signal = filtfilt(b, a, signal_data)
    return filtered_signal

# Create signal with ONLY baseline drift
_, clean_drift = generate_clean_ecg(duration=10, sampling_rate=sampling_rate)
noisy_drift = add_noise(clean_drift, sampling_rate=sampling_rate,
                        high_freq_noise=0,
                        baseline_drift=0.3,  # Strong baseline drift
                        powerline=False,
                        artifacts=False)

# Apply high-pass filter
filtered_drift = apply_highpass_filter(noisy_drift, cutoff_freq=0.5, sampling_rate=sampling_rate)

time_drift = np.arange(len(clean_drift)) / sampling_rate

# Visualize
fig, axes = plt.subplots(3, 1, figsize=(16, 11))

# Clean
axes[0].plot(time_drift, clean_drift, 'g-', linewidth=2, label='Clean (Original)')
axes[0].set_ylabel('Voltage (mV)', fontsize=11)
axes[0].set_title('✅ Original Clean Signal', fontsize=13, fontweight='bold', color='green')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# With drift
axes[1].plot(time_drift, noisy_drift, 'r-', linewidth=1.5, alpha=0.8, label='With Baseline Drift')
axes[1].set_ylabel('Voltage (mV)', fontsize=11)
axes[1].set_title('❌ With Baseline Drift (Breathing & Movement)', fontsize=13, fontweight='bold', color='red')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

# Filtered
axes[2].plot(time_drift, filtered_drift, 'b-', linewidth=2, label='After High-Pass Filter')
axes[2].plot(time_drift, clean_drift, 'g--', linewidth=1.5, alpha=0.5, label='Original (reference)')
axes[2].set_xlabel('Time (seconds)', fontsize=11)
axes[2].set_ylabel('Voltage (mV)', fontsize=11)
axes[2].set_title('✅ After High-Pass Filter (0.5 Hz cutoff)', fontsize=13, fontweight='bold', color='blue')
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()

print("\n🎉 Excellent! The baseline is now stable!")
print("   The high-pass filter removed the slow wandering drift.")

---
# 🔧 Tool #3: Band-Pass Filter (Best of Both Worlds)

## What is a Band-Pass Filter?

A **band-pass filter** combines high-pass and low-pass filtering:
- Removes low frequencies (baseline drift)
- Removes high frequencies (noise)
- **Keeps only the middle band** where your signal lives!

## When to Use It

Band-pass is the most common filter for biosignals because real signals face BOTH problems.

## Typical Bands

- **ECG**: 0.5-40 Hz
- **EEG**: 0.5-50 Hz  
- **EMG**: 20-450 Hz

Let's create the ultimate cleaning filter!

In [ ]:
def apply_bandpass_filter(signal_data, lowcut, highcut, sampling_rate=500, order=4):
    """
    Apply a band-pass Butterworth filter
    
    lowcut: low cutoff frequency (removes below this)
    highcut: high cutoff frequency (removes above this)
    """
    nyquist = sampling_rate / 2
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band', analog=False)
    filtered_signal = filtfilt(b, a, signal_data)
    return filtered_signal

# Create signal with BOTH drift AND noise
_, clean_both = generate_clean_ecg(duration=10, sampling_rate=sampling_rate)
noisy_both = add_noise(clean_both, sampling_rate=sampling_rate,
                       high_freq_noise=0.12,
                       baseline_drift=0.25,
                       powerline=False,  # We'll handle this separately
                       artifacts=False)

# Apply band-pass filter (0.5-40 Hz for ECG)
filtered_both = apply_bandpass_filter(noisy_both, lowcut=0.5, highcut=40, 
                                      sampling_rate=sampling_rate)

time_both = np.arange(len(clean_both)) / sampling_rate

# Compare all three
fig, axes = plt.subplots(3, 1, figsize=(16, 11))

# Clean
axes[0].plot(time_both, clean_both, 'g-', linewidth=2, label='Clean (Original)')
axes[0].set_ylabel('Voltage (mV)', fontsize=11)
axes[0].set_title('✅ Original Clean Signal', fontsize=13, fontweight='bold', color='green')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Noisy (drift + HF noise)
axes[1].plot(time_both, noisy_both, 'r-', linewidth=1.5, alpha=0.7, label='Drift + HF Noise')
axes[1].set_ylabel('Voltage (mV)', fontsize=11)
axes[1].set_title('❌ With Baseline Drift AND High-Frequency Noise', fontsize=13, fontweight='bold', color='red')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

# Band-pass filtered
axes[2].plot(time_both, filtered_both, 'b-', linewidth=2, label='After Band-Pass Filter')
axes[2].plot(time_both, clean_both, 'g--', linewidth=1.5, alpha=0.5, label='Original (reference)')
axes[2].set_xlabel('Time (seconds)', fontsize=11)
axes[2].set_ylabel('Voltage (mV)', fontsize=11)
axes[2].set_title('✅ After Band-Pass Filter (0.5-40 Hz)', fontsize=13, fontweight='bold', color='blue')
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()

# Calculate Signal-to-Noise Ratio improvement
noise_before = noisy_both - clean_both
noise_after = filtered_both - clean_both
snr_improvement = 10 * np.log10(np.var(noise_before) / np.var(noise_after))

print("\n🎉 Band-pass filter cleaned BOTH types of noise!")
print(f"   SNR improvement: {snr_improvement:.1f} dB")
print("\n💡 This is why band-pass filtering is the go-to method for biosignals!")

---
# 🔧 Tool #4: Notch Filter (Remove Power Line Interference)

## What is a Notch Filter?

A **notch filter** removes a very specific frequency while keeping everything else. It's like a surgical strike!

## The Power Line Problem

Electrical wiring creates interference at:
- **50 Hz** in Europe, Africa, Asia, Australia
- **60 Hz** in USA, Canada, parts of South America

This creates a constant "hum" in recordings.

## How It Works

The notch filter creates a narrow "notch" (gap) in the frequency response:
- Removes just 60 Hz (or 50 Hz)
- Keeps everything else untouched

Let's eliminate that annoying power line hum!

In [ ]:
def apply_notch_filter(signal_data, notch_freq=60, sampling_rate=500, quality_factor=30):
    """
    Apply a notch filter to remove specific frequency
    
    notch_freq: frequency to remove (50 or 60 Hz typically)
    quality_factor: higher = narrower notch (more selective)
    """
    nyquist = sampling_rate / 2
    freq = notch_freq / nyquist
    b, a = iirnotch(freq, quality_factor)
    filtered_signal = filtfilt(b, a, signal_data)
    return filtered_signal

# Create signal with power line interference
_, clean_power = generate_clean_ecg(duration=10, sampling_rate=sampling_rate)
noisy_power = add_noise(clean_power, sampling_rate=sampling_rate,
                        high_freq_noise=0.05,
                        baseline_drift=0,
                        powerline=True,  # 60 Hz interference!
                        artifacts=False)

# Apply notch filter at 60 Hz
filtered_power = apply_notch_filter(noisy_power, notch_freq=60, sampling_rate=sampling_rate)

time_power = np.arange(len(clean_power)) / sampling_rate

# Visualize
fig, axes = plt.subplots(3, 1, figsize=(16, 11))

# Clean
zoom = slice(0, 1500)  # Zoom to see the 60 Hz ripple
axes[0].plot(time_power[zoom], clean_power[zoom], 'g-', linewidth=2, label='Clean')
axes[0].set_ylabel('Voltage (mV)', fontsize=11)
axes[0].set_title('✅ Original Clean Signal', fontsize=13, fontweight='bold', color='green')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# With 60 Hz interference
axes[1].plot(time_power[zoom], noisy_power[zoom], 'r-', linewidth=1.5, alpha=0.8, label='With 60 Hz Interference')
axes[1].set_ylabel('Voltage (mV)', fontsize=11)
axes[1].set_title('❌ With 60 Hz Power Line Interference (See the Regular Ripple)', 
                 fontsize=13, fontweight='bold', color='red')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

# After notch filter
axes[2].plot(time_power[zoom], filtered_power[zoom], 'b-', linewidth=2, label='After 60 Hz Notch Filter')
axes[2].plot(time_power[zoom], clean_power[zoom], 'g--', linewidth=1.5, alpha=0.5, label='Original (reference)')
axes[2].set_xlabel('Time (seconds)', fontsize=11)
axes[2].set_ylabel('Voltage (mV)', fontsize=11)
axes[2].set_title('✅ After Notch Filter - 60 Hz Removed!', fontsize=13, fontweight='bold', color='blue')
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()

print("\n🎉 The 60 Hz ripple is gone!")
print("   The notch filter surgically removed just the power line frequency.")
print("\n💡 Note: If you're in Europe, use notch_freq=50 instead!")

---
# 🔧 Tool #5: Artifact Removal (Clean Up Big Spikes)

## What Are Artifacts?

**Artifacts** are large, sudden distortions caused by:
- Eye blinks (in EEG)
- Movement (muscle activity)
- Electrode contact issues
- Electrical interference spikes

## Detection Strategy

We can detect artifacts by:
1. Finding values much larger than normal (outliers)
2. Looking for sudden jumps in amplitude
3. Checking for impossible values

## Removal Strategy

Once detected, we can:
- Replace with interpolated values
- Mark segments as "bad" and exclude from analysis
- Use median filtering for robust smoothing

Let's implement artifact detection and removal!

In [ ]:
def detect_and_remove_artifacts(signal_data, threshold_std=3.0, window_size=50):
    """
    Detect and remove artifacts using threshold-based detection
    
    threshold_std: number of standard deviations to consider artifact
    window_size: size of window for median filtering
    """
    # Calculate threshold based on signal statistics
    mean_val = np.mean(signal_data)
    std_val = np.std(signal_data)
    threshold = threshold_std * std_val
    
    # Detect artifacts (values far from mean)
    artifacts = np.abs(signal_data - mean_val) > threshold
    
    # Create cleaned signal
    cleaned_signal = signal_data.copy()
    
    # Replace artifacts with median of surrounding values
    artifact_indices = np.where(artifacts)[0]
    for idx in artifact_indices:
        # Get window around artifact
        start = max(0, idx - window_size // 2)
        end = min(len(signal_data), idx + window_size // 2)
        
        # Get non-artifact values in window
        window = signal_data[start:end]
        window_mask = ~artifacts[start:end]
        
        if np.any(window_mask):
            # Replace with median of good values
            cleaned_signal[idx] = np.median(window[window_mask])
    
    return cleaned_signal, artifacts, artifact_indices

# Generate signal with artifacts
_, clean_art = generate_clean_ecg(duration=10, sampling_rate=sampling_rate)
noisy_art = add_noise(clean_art, sampling_rate=sampling_rate,
                      high_freq_noise=0.08,
                      baseline_drift=0.1,
                      powerline=False,
                      artifacts=True)  # Add artifacts!

# Remove artifacts
cleaned_art, artifact_mask, artifact_idx = detect_and_remove_artifacts(noisy_art, threshold_std=2.5)

time_art = np.arange(len(clean_art)) / sampling_rate

# Visualize
fig, axes = plt.subplots(3, 1, figsize=(16, 11))

# Clean original
axes[0].plot(time_art, clean_art, 'g-', linewidth=2, label='Clean')
axes[0].set_ylabel('Voltage (mV)', fontsize=11)
axes[0].set_title('✅ Original Clean Signal', fontsize=13, fontweight='bold', color='green')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# With artifacts (highlighted)
axes[1].plot(time_art, noisy_art, 'r-', linewidth=1.5, alpha=0.6, label='With Artifacts')
axes[1].scatter(time_art[artifact_mask], noisy_art[artifact_mask], 
               c='red', s=50, marker='x', linewidths=2, label='Detected Artifacts', zorder=5)
axes[1].set_ylabel('Voltage (mV)', fontsize=11)
axes[1].set_title(f'❌ With Artifacts ({len(artifact_idx)} detected)', 
                 fontsize=13, fontweight='bold', color='red')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

# After artifact removal
axes[2].plot(time_art, cleaned_art, 'b-', linewidth=2, label='After Artifact Removal')
axes[2].plot(time_art, clean_art, 'g--', linewidth=1.5, alpha=0.5, label='Original (reference)')
axes[2].set_xlabel('Time (seconds)', fontsize=11)
axes[2].set_ylabel('Voltage (mV)', fontsize=11)
axes[2].set_title('✅ After Artifact Removal', fontsize=13, fontweight='bold', color='blue')
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"\n🎉 Detected and removed {len(artifact_idx)} artifacts!")
print(f"   Artifact locations (seconds): {time_art[artifact_idx[:5]]:.2f}... (showing first 5)")
print("\n💡 The signal now looks much cleaner without those big spikes!")

---
# 🎯 The Complete Cleaning Pipeline

Now let's put it all together! We'll create a complete preprocessing pipeline that:
1. Removes artifacts
2. Applies band-pass filter (removes drift + HF noise)
3. Applies notch filter (removes 60 Hz)

This is what professionals do in real signal processing!

In [ ]:
def complete_preprocessing_pipeline(signal_data, sampling_rate=500, 
                                    lowcut=0.5, highcut=40, notch_freq=60):
    """
    Complete signal cleaning pipeline
    
    Steps:
    1. Artifact detection and removal
    2. Band-pass filtering (0.5-40 Hz for ECG)
    3. Notch filtering (remove 60 Hz power line)
    """
    print("🔧 Starting preprocessing pipeline...\n")
    
    # Step 1: Remove artifacts
    print("Step 1/3: Detecting and removing artifacts...")
    cleaned, artifacts, art_idx = detect_and_remove_artifacts(signal_data, threshold_std=2.5)
    print(f"   ✅ Removed {len(art_idx)} artifacts\n")
    
    # Step 2: Band-pass filter
    print(f"Step 2/3: Applying band-pass filter ({lowcut}-{highcut} Hz)...")
    cleaned = apply_bandpass_filter(cleaned, lowcut, highcut, sampling_rate)
    print(f"   ✅ Removed baseline drift and high-frequency noise\n")
    
    # Step 3: Notch filter
    print(f"Step 3/3: Applying notch filter ({notch_freq} Hz)...")
    cleaned = apply_notch_filter(cleaned, notch_freq, sampling_rate)
    print(f"   ✅ Removed {notch_freq} Hz power line interference\n")
    
    print("🎉 Preprocessing complete!\n")
    
    return cleaned

# Create the WORST possible signal (all noise types!)
print("Creating extremely noisy ECG signal...\n")
_, pristine_ecg = generate_clean_ecg(duration=15, heart_rate=72, sampling_rate=sampling_rate)
horrible_ecg = add_noise(pristine_ecg, sampling_rate=sampling_rate,
                         high_freq_noise=0.15,   # Lots of noise
                         baseline_drift=0.3,      # Strong drift
                         powerline=True,          # 60 Hz hum
                         artifacts=True)          # Random spikes

print("="*70)
# Apply complete pipeline
beautiful_ecg = complete_preprocessing_pipeline(horrible_ecg, sampling_rate=sampling_rate)
print("="*70)

# Visualize the transformation
time_complete = np.arange(len(pristine_ecg)) / sampling_rate

fig, axes = plt.subplots(3, 1, figsize=(16, 12))

# Original pristine
axes[0].plot(time_complete, pristine_ecg, 'g-', linewidth=2, label='Original Clean Signal')
axes[0].set_ylabel('Voltage (mV)', fontsize=12)
axes[0].set_title('✅ Original Clean ECG (Ground Truth)', fontsize=14, fontweight='bold', color='green')
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=11)

# Horrible noisy version
axes[1].plot(time_complete, horrible_ecg, 'r-', linewidth=1.5, alpha=0.7, label='All Types of Noise')
axes[1].set_ylabel('Voltage (mV)', fontsize=12)
axes[1].set_title('❌ Extremely Noisy ECG (Drift + HF Noise + 60Hz + Artifacts)', 
                 fontsize=14, fontweight='bold', color='red')
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=11)

# After complete cleaning
axes[2].plot(time_complete, beautiful_ecg, 'b-', linewidth=2, label='After Complete Pipeline')
axes[2].plot(time_complete, pristine_ecg, 'g--', linewidth=1.5, alpha=0.5, label='Original (reference)')
axes[2].set_xlabel('Time (seconds)', fontsize=12)
axes[2].set_ylabel('Voltage (mV)', fontsize=12)
axes[2].set_title('✅ After Complete Preprocessing Pipeline - Crystal Clear!', 
                 fontsize=14, fontweight='bold', color='blue')
axes[2].grid(True, alpha=0.3)
axes[2].legend(fontsize=11)

plt.tight_layout()
plt.show()

# Calculate quality metrics
noise_original = horrible_ecg - pristine_ecg
noise_cleaned = beautiful_ecg - pristine_ecg

snr_before = 10 * np.log10(np.var(pristine_ecg) / np.var(noise_original))
snr_after = 10 * np.log10(np.var(pristine_ecg) / np.var(noise_cleaned))
improvement = snr_after - snr_before

print("\n" + "="*70)
print("📊 PREPROCESSING RESULTS")
print("="*70)
print(f"SNR before cleaning: {snr_before:.2f} dB (very noisy)")
print(f"SNR after cleaning:  {snr_after:.2f} dB (much cleaner!)")
print(f"\n🎉 Signal quality improvement: {improvement:.2f} dB")
print("\n💡 The cleaned signal closely matches the original clean version!")
print("   This is professional-grade signal preprocessing!")

---
# 🧪 Your Turn - Signal Cleaning Challenges!

Test your preprocessing skills!

## Exercise 1: Optimize Filter Parameters 🎯

Find the best filter parameters for cleaning a specific type of noise!

**Your task**: Experiment with different cutoff frequencies to find optimal values.

In [ ]:
# Exercise 1: Find optimal low-pass cutoff
_, test_clean = generate_clean_ecg(duration=10, sampling_rate=500)
test_noisy = add_noise(test_clean, sampling_rate=500, 
                       high_freq_noise=0.15, baseline_drift=0, 
                       powerline=False, artifacts=False)

# Try different cutoff frequencies
cutoffs_to_test = [20, 30, 40, 50, 60]
snr_results = []

for cutoff in cutoffs_to_test:
    filtered = apply_lowpass_filter(test_noisy, cutoff_freq=cutoff, sampling_rate=500)
    noise = filtered - test_clean
    snr = 10 * np.log10(np.var(test_clean) / np.var(noise))
    snr_results.append(snr)

# Plot results
plt.figure(figsize=(12, 6))
plt.plot(cutoffs_to_test, snr_results, 'o-', linewidth=2, markersize=10, color='blue')
plt.xlabel('Cutoff Frequency (Hz)', fontsize=12)
plt.ylabel('SNR (dB)', fontsize=12)
plt.title('🎯 Low-Pass Filter Optimization: Find the Best Cutoff!', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

# Mark the best
best_idx = np.argmax(snr_results)
best_cutoff = cutoffs_to_test[best_idx]
best_snr = snr_results[best_idx]
plt.plot(best_cutoff, best_snr, 'r*', markersize=20, label=f'Best: {best_cutoff} Hz')
plt.legend(fontsize=11)
plt.show()

print(f"\n🏆 Optimal cutoff frequency: {best_cutoff} Hz")
print(f"   Achieves SNR of {best_snr:.2f} dB")
print(f"\n💡 This is the sweet spot - removes noise without removing signal!")

## Exercise 2: Build a Real-Time Quality Monitor 📊

Create a function that rates signal quality before and after cleaning!

**Your task**: Complete the quality assessment function.

In [ ]:
def assess_signal_quality(signal_data, reference_clean=None):
    """
    Assess signal quality metrics
    
    Returns quality score and metrics
    """
    metrics = {}
    
    # 1. Amplitude variability (lower is better for noise)
    metrics['std'] = np.std(signal_data)
    
    # 2. High-frequency content (FFT-based)
    fft_vals = np.abs(fft(signal_data))
    high_freq_power = np.sum(fft_vals[len(fft_vals)//4:])  # Upper 75% of spectrum
    total_power = np.sum(fft_vals)
    metrics['hf_ratio'] = high_freq_power / total_power
    
    # 3. If we have reference, calculate SNR
    if reference_clean is not None:
        noise = signal_data - reference_clean
        snr = 10 * np.log10(np.var(reference_clean) / np.var(noise))
        metrics['snr'] = snr
    
    # Quality score (0-100)
    # Based on HF ratio (lower is better)
    score = max(0, min(100, 100 - metrics['hf_ratio'] * 300))
    
    return score, metrics

# Test on noisy and cleaned signals
_, ref = generate_clean_ecg(duration=10, sampling_rate=500)
noisy = add_noise(ref, sampling_rate=500, high_freq_noise=0.15, 
                  baseline_drift=0.2, powerline=True, artifacts=True)
cleaned = complete_preprocessing_pipeline(noisy, sampling_rate=500)

score_noisy, metrics_noisy = assess_signal_quality(noisy, ref)
score_cleaned, metrics_cleaned = assess_signal_quality(cleaned, ref)

print("\n" + "="*70)
print("📊 SIGNAL QUALITY ASSESSMENT")
print("="*70)

print("\n❌ BEFORE Preprocessing:")
print(f"   Quality Score: {score_noisy:.1f}/100")
print(f"   SNR: {metrics_noisy['snr']:.2f} dB")
print(f"   HF Noise Ratio: {metrics_noisy['hf_ratio']:.3f}")

print("\n✅ AFTER Preprocessing:")
print(f"   Quality Score: {score_cleaned:.1f}/100")
print(f"   SNR: {metrics_cleaned['snr']:.2f} dB")
print(f"   HF Noise Ratio: {metrics_cleaned['hf_ratio']:.3f}")

improvement_pct = ((score_cleaned - score_noisy) / score_noisy) * 100
print(f"\n🎉 Quality improvement: {improvement_pct:.1f}%")

---
# 🎉 Congratulations, Signal Cleaning Expert!

## What You Mastered Today

Incredible work! You now know professional signal preprocessing:

### 🔧 Filtering Techniques
- ✅ **Low-pass filters** - Remove high-frequency noise
- ✅ **High-pass filters** - Remove baseline drift
- ✅ **Band-pass filters** - Keep only desired frequency range
- ✅ **Notch filters** - Remove specific interference (50/60 Hz)

### 🧹 Advanced Cleaning
- ✅ Artifact detection and removal
- ✅ Complete preprocessing pipelines
- ✅ Quality assessment metrics
- ✅ SNR calculation and optimization

### 📊 Understanding Noise
- ✅ High-frequency noise sources
- ✅ Baseline drift causes
- ✅ Power line interference
- ✅ Artifact types and handling

### 🛠️ Practical Skills
- ✅ Choosing appropriate filter parameters
- ✅ Optimizing preprocessing pipelines
- ✅ Comparing before/after results
- ✅ Professional-grade signal processing

## 🚀 What's Next?

**Next Notebook**: `05_Pattern_Recognition.ipynb` - Extract features and recognize patterns!

## 💡 Real-World Applications

These preprocessing skills are essential for:
- 🏥 Medical device development
- 📱 Wearable health monitors
- 🔬 Research signal analysis
- 🤖 Machine learning preprocessing
- 📊 Clinical diagnosis systems

---

### 📝 Key Takeaways

> **Preprocessing is Critical**: Real biosignals are never clean - preprocessing is essential!
>
> **Band-Pass is King**: Most biosignal applications use band-pass filtering
>
> **Pipeline Approach**: Combine multiple techniques for best results
>
> **Always Validate**: Compare cleaned signals to ground truth when possible

**You're now ready to handle messy, real-world biosignals!** 🧹✨

---

*Made with ❤️ for curious minds by the Delta-Predictive-Biosensing team*